In [ ]:
# -*- coding: utf-8 -*-
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or
# implied.
# See the License for the specific language governing permissions and
# limitations under the License.


# Imports

In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, GridSearchCV
from sklearn.decomposition import TruncatedSVD
import os
import wandb
from scipy.stats import uniform, loguniform, randint
import sys
sys.path.append(os.path.join(os.getcwd(), '..'))
from util.preprocessing import TweetPreprocessor 
from util.logger import wandb_log_search_results, wandb_save_model
from joblib import cpu_count
from sklearn.model_selection import train_test_split

In [ ]:
n_jobs = max(1, cpu_count() - 1)

# Consts

In [ ]:
MODEL_DIR = os.path.join(os.getcwd(), 'models')
DATASETS_DIR = os.path.join(os.getcwd(), 'datasets')
ESTIMATORS = [
    LinearSVC(
        random_state=int(os.getenv("RANDOM_SEED", 880055535)),
    ),
    RandomForestClassifier(
        random_state=int(os.getenv("RANDOM_SEED", 880055535)),
    ),
    LogisticRegression(
        random_state=int(os.getenv("RANDOM_SEED", 880055535)),
    ),
]
ESTIMATOR_NUMBER = 0 # 0 - LinearSVC, 1 - RandomForestClassifier, 2 - LogisticRegression
ESTIMATORS_PARAMS = [
    {
        "clf__C": loguniform(1e-3, 1e3),
    },
    {
        "clf__n_estimators": randint(50, 600),
        "clf__max_depth": randint(5, 50),
        "clf__min_samples_split": randint(2, 20),
        "clf__min_samples_leaf": randint(1, 20),
        "clf__max_features": ["sqrt", "log2"],
    },
    {
        "clf__C": loguniform(1e-3, 1e3),
        "clf__solver": ["lbfgs", "liblinear"],
        "clf__penalty": ["l2"],
        "clf__class_weight": [None, "balanced"],
        "clf__max_iter": randint(100, 3000),
    },
]
ESTIMATOR = ESTIMATORS[ESTIMATOR_NUMBER]
ESTIMATOR_PARAMS = ESTIMATORS_PARAMS[ESTIMATOR_NUMBER] 

# Dataset

In [ ]:
train_df = pd.read_csv(os.path.join(DATASETS_DIR, "one_tweet_dataset_train.csv"))
val_df = pd.read_csv(os.path.join(DATASETS_DIR, "one_tweet_dataset_val.csv"))
train_df = pd.concat([train_df, val_df], ignore_index=True)

In [ ]:
x_train, y_train = train_df["text"], train_df["gender_label"].map({'F': 0, 'M': 1})

for hyperparam search there enough 40000 samples, because searching for best hyperparameters on full dataset is very time consuming.

In [ ]:
x_train, _, y_train, _ = train_test_split(x_train, y_train, test_size=0.3, random_state=int(os.getenv("RANDOM_SEED", 880055535)), stratify=y_train)

# Init wandb

In [ ]:
wandbToken = os.getenv("WANDB_TOKEN")
if not wandbToken:
    raise ValueError("Please set the WANDB_TOKEN environment variable to log results to Weights & Biases.")
wandb.login(key=wandbToken)

# Randomized search

In [ ]:
cv = StratifiedKFold(
    n_splits=4,
    shuffle=True,
    random_state=int(os.getenv("RANDOM_SEED", 880055535)),
)

## TF-idf 

In [ ]:
texts = TweetPreprocessor().fit_transform(x_train) # type: ignore

### Word

In [ ]:
ngram_ranges_word = [(1, 2), (1, 3)]

rnd_params_tfidf_word = {
    "tfidf_word__use_idf": [True, False],
    "tfidf_word__sublinear_tf": [True, False],
    "tfidf_word__norm": ["l1", "l2"],
    "tfidf_word__max_df": uniform(0.6, 0.4),
    "tfidf_word__min_df": uniform(0.001, 0.4),
    "tfidf_word__max_features": randint(5000, 120000),
    "tfidf_word__ngram_range": ngram_ranges_word,
}
rnd_params_tfidf_word

In [ ]:
rnd_search_tfidf_word = RandomizedSearchCV(
    Pipeline([
        ("tfidf_word", TfidfVectorizer(analyzer="word")), # type: ignore
        ("clf", ESTIMATOR), # type: ignore
    ]),
    rnd_params_tfidf_word, 
    n_iter=100,
    cv=cv,
    scoring="f1_macro",
    n_jobs=n_jobs,
    verbose=5,
    random_state=int(os.getenv("RANDOM_SEED", 880055535))
)

In [ ]:
rnd_search_tfidf_word.fit(texts, y_train)

In [ ]:
rnd_params_after_tfidf_word = rnd_search_tfidf_word.best_params_
rnd_params_after_tfidf_word

In [ ]:
wandb_log_search_results(rnd_search_tfidf_word, "hyperparams_rnd_search_word", "hyperparams_rnd_search")

### Char

In [ ]:
ngram_ranges_char = [(2, 4), (3, 5), (4, 6)]

rnd_params_tfidf_char = {
    "tfidf_char__use_idf": [True, False],
    "tfidf_char__sublinear_tf": [True, False],
    "tfidf_char__norm": ["l1", "l2"],
    "tfidf_char__max_df": uniform(0.6, 0.4),
    "tfidf_char__min_df": uniform(0.001, 0.4),
    "tfidf_char__max_features": randint(5000, 120000),
    "tfidf_char__ngram_range": ngram_ranges_char,
}
rnd_params_tfidf_char

In [ ]:
rnd_search_tfidf_char = RandomizedSearchCV(
    Pipeline([
        ("tfidf_char", TfidfVectorizer(analyzer="char")), # type: ignore
        ("clf", ESTIMATOR), # type: ignore
    ]),
    rnd_params_tfidf_char,
    n_iter=100,
    cv=cv,
    scoring="f1_macro",
    n_jobs=n_jobs,
    verbose=5,
    random_state=int(os.getenv("RANDOM_SEED", 880055535))
)

In [ ]:
rnd_search_tfidf_char.fit(texts, y_train)

In [ ]:
rnd_params_after_tfidf_char = rnd_search_tfidf_char.best_params_
rnd_params_after_tfidf_char

In [ ]:
wandb_log_search_results(rnd_search_tfidf_char, "hyperparams_rnd_search_char", "hyperparams_rnd_search")

## TF-IDF char + word randomized search

In [ ]:
rnd_tfidf_pipeline_char_word = Pipeline([
    ("features", FeatureUnion([
        ("tfidf_char", TfidfVectorizer(analyzer="char")), # type: ignore
        ("tfidf_word", TfidfVectorizer(analyzer="word")), # type: ignore
    ])),
])
rnd_tfidf_pipeline_char_word.set_params(
    **{f"features__{k}": v for k, v in rnd_params_after_tfidf_word.items()},
    **{f"features__{k}": v for k, v in rnd_params_after_tfidf_char.items()},
)

In [ ]:
X_tfidf_char_word = rnd_tfidf_pipeline_char_word.fit_transform(texts, y_train)
X_tfidf_char_word.shape

## SVD

In [ ]:
rnd_svd_pipeline = Pipeline([
    ("svd", TruncatedSVD(random_state=int(os.getenv("RANDOM_SEED", 880055535)))), # type: ignore
    ("clf", ESTIMATOR), # type: ignore
])

In [ ]:
rnd_svd_params = {
    "svd__n_components": randint(100, 500),
}
rnd_svd_params

In [ ]:
rnd_svd_search = RandomizedSearchCV(
    rnd_svd_pipeline,
    rnd_svd_params, 
    n_iter=10,
    cv=cv,
    scoring="f1_macro",
    n_jobs=n_jobs,
    verbose=5,
    random_state=int(os.getenv("RANDOM_SEED", 880055535))
)

In [ ]:
rnd_svd_search.fit(X_tfidf_char_word, y_train)

In [ ]:
rnd_params_after_svd = rnd_svd_search.best_params_
rnd_params_after_svd

In [ ]:
wandb_log_search_results(rnd_svd_search, "hyperparams_rnd_search_svd", "hyperparams_rnd_search")

In [ ]:
after_grid_svd_pipeline = Pipeline([
    ("svd", TruncatedSVD(random_state=int(os.getenv("RANDOM_SEED", 880055535)))), # type: ignore
])
after_grid_svd_pipeline.set_params(
    **rnd_params_after_svd,
)

In [ ]:
X_svd_rnd = after_grid_svd_pipeline.fit_transform(X_tfidf_char_word, y_train)

## CLF

In [ ]:
rnd_clf_pipeline_svc = Pipeline([
    ("clf", ESTIMATOR), # type: ignore
])

In [ ]:
rnd_clf_params = ESTIMATOR_PARAMS
rnd_clf_params

In [ ]:
rnd_clf_search = RandomizedSearchCV(
    rnd_clf_pipeline_svc,
    rnd_clf_params,
    n_iter=50,
    cv=cv,
    scoring="f1_macro",
    n_jobs=n_jobs,
    verbose=5,
    random_state=int(os.getenv("RANDOM_SEED", 880055535))
)

In [ ]:
rnd_clf_search.fit(X_svd_rnd, y_train)

In [ ]:
wandb_log_search_results(rnd_clf_search, "hyperparams_rnd_search_clf_svc", "hyperparams_rnd_search")

In [ ]:
wandb_save_model(
    rnd_clf_search.best_estimator_,
    os.path.join(MODEL_DIR, "best_model_after_random_search_svc"),
)

# Grid search

In [ ]:
best_rnd_params_clf = rnd_clf_search.best_params_

## TF-idf

### Word

In [ ]:
def make_range(value, pct=0.05, cast_int=False):
    factors = [1 - pct, 1, 1 + pct]

    candidates = [value * f for f in factors]

    if cast_int:
        candidates = [int(round(c)) for c in candidates]

    uniq = []
    for c in candidates:
        if c not in uniq:
            uniq.append(c)

    if cast_int and len(uniq) == 1:
        uniq = [c for c in [uniq[0] - 1, uniq[0], uniq[0] + 1] if c >= 1]

    return uniq

In [ ]:
def make_range_from_params(params: dict, pct=0.1):
    range_params = {}
    for k, v in params.items():
        if isinstance(v, (int, float, np.integer, np.floating)):
            range_params[k] = make_range(v, pct=pct, cast_int=isinstance(v, (int, np.integer)))
        else:
            range_params[k] = [v]
    return range_params

In [ ]:
grid_tfidf_pipeline_word = Pipeline([
    ("tfidf_word", TfidfVectorizer(analyzer="word")), # type: ignore
    ("clf", ESTIMATOR), # type: ignore
])
grid_tfidf_pipeline_word.set_params(
    **rnd_params_after_tfidf_word,
    **best_rnd_params_clf,
)

In [ ]:
grid_tfidf_params_word = {
    "tfidf_word__use_idf": [rnd_params_after_tfidf_word["tfidf_word__use_idf"]],
    "tfidf_word__sublinear_tf": [rnd_params_after_tfidf_word["tfidf_word__sublinear_tf"]],
    "tfidf_word__norm": [rnd_params_after_tfidf_word["tfidf_word__norm"]],
    "tfidf_word__ngram_range": [rnd_params_after_tfidf_word["tfidf_word__ngram_range"]],
    "tfidf_word__max_df": make_range(rnd_params_after_tfidf_word["tfidf_word__max_df"]),
    "tfidf_word__min_df": make_range(rnd_params_after_tfidf_word["tfidf_word__min_df"]),
    "tfidf_word__max_features": make_range(
        rnd_params_after_tfidf_word["tfidf_word__max_features"], cast_int=True
    ),
}
grid_tfidf_params_word

In [ ]:
grid_tfidf_search_word = GridSearchCV(
    Pipeline([
        ("tfidf_word", TfidfVectorizer(analyzer="word")), # type: ignore
        ("clf", ESTIMATOR),
    ]),
    grid_tfidf_params_word,
    cv=cv,
    scoring="f1_macro",
    n_jobs=n_jobs,
    verbose=5,
)

In [ ]:
grid_tfidf_search_word.fit(texts, y_train)

In [ ]:
best_tfidf_word = grid_tfidf_search_word.best_params_
best_tfidf_word

In [ ]:
wandb_log_search_results(grid_tfidf_search_word, "hyperparams_grid_search_word", "hyperparams_grid_search")

### Char

In [ ]:
grid_tfidf_pipeline_char = Pipeline([
    ("tfidf_char", TfidfVectorizer(analyzer="char")), # type: ignore
    ("clf", ESTIMATOR), # type: ignore
])
grid_tfidf_pipeline_char.set_params(
    **rnd_params_after_tfidf_char,
    **best_rnd_params_clf,
)

In [ ]:
grid_tfidf_params_char = {
    "tfidf_char__use_idf": [rnd_params_after_tfidf_char["tfidf_char__use_idf"]],
    "tfidf_char__sublinear_tf": [rnd_params_after_tfidf_char["tfidf_char__sublinear_tf"]],
    "tfidf_char__norm": [rnd_params_after_tfidf_char["tfidf_char__norm"]],
    "tfidf_char__ngram_range": [rnd_params_after_tfidf_char["tfidf_char__ngram_range"]],
    "tfidf_char__max_df": make_range(rnd_params_after_tfidf_char["tfidf_char__max_df"]),
    "tfidf_char__min_df": make_range(rnd_params_after_tfidf_char["tfidf_char__min_df"]),
    "tfidf_char__max_features": make_range(
        rnd_params_after_tfidf_char["tfidf_char__max_features"], cast_int=True
    ),
}
grid_tfidf_params_char

In [ ]:
grid_tfidf_search_char = GridSearchCV(
    grid_tfidf_pipeline_char,
    grid_tfidf_params_char,
    cv=cv,
    scoring="f1_macro",
    n_jobs=n_jobs,
    verbose=5,
)

In [ ]:
grid_tfidf_search_char.fit(texts, y_train)

In [ ]:
best_tfidf_char = grid_tfidf_search_char.best_params_
best_tfidf_char

In [ ]:
wandb_log_search_results(grid_tfidf_search_char, "hyperparams_grid_search_char", "hyperparams_grid_search")

## TF-IDF char + word grid search

In [ ]:
grid_tfidf_pipeline_char_word = Pipeline([
    ("features", FeatureUnion([
        ("tfidf_char", TfidfVectorizer(analyzer="char")), # type: ignore
        ("tfidf_word", TfidfVectorizer(analyzer="word")), # type: ignore
    ])),
])
grid_tfidf_pipeline_char_word.set_params(
    **{f"features__{k}": v for k, v in best_tfidf_word.items()},
    **{f"features__{k}": v for k, v in best_tfidf_char.items()},
)

In [ ]:
X_tfidf_char_word = grid_tfidf_pipeline_char_word.fit_transform(texts, y_train)
X_tfidf_char_word.shape

## SVD

In [ ]:
grid_svd_pipeline = Pipeline([
    ("svd", TruncatedSVD(random_state=int(os.getenv("RANDOM_SEED", 880055535)))), # type: ignore
    ("clf", ESTIMATOR), # type: ignore
])

grid_svd_pipeline.set_params(
    **best_rnd_params_clf
)

In [ ]:
grid_svd_params = {
    "svd__n_components": make_range(rnd_params_after_svd["svd__n_components"], pct=0.1, cast_int=True),
}
grid_svd_params

In [ ]:
grid_svd_search = GridSearchCV(
    grid_svd_pipeline,
    grid_svd_params,
    cv=cv,
    scoring="f1_macro",
    n_jobs=n_jobs,
    verbose=5,
)

grid_svd_search.fit(X_tfidf_char_word, y_train)

In [ ]:
best_svd_params = grid_svd_search.best_params_

In [ ]:
wandb_log_search_results(grid_svd_search, "hyperparams_grid_search_svd", "hyperparams_grid_search")

In [ ]:
after_grid_svd_pipeline = Pipeline([
    ("svd", TruncatedSVD(random_state=int(os.getenv("RANDOM_SEED", 880055535)))), # type: ignore
])
after_grid_svd_pipeline.set_params(
    **best_svd_params,
)

In [ ]:
X_svd_grid = after_grid_svd_pipeline.fit_transform(X_tfidf_char_word, y_train)

## CLF

In [ ]:
grid_clf_pipeline = Pipeline([
    ("clf", ESTIMATOR),
])

In [ ]:
grid_clf_params = make_range_from_params(
    {k: v for k, v in best_rnd_params_clf.items() if k.startswith("clf__")}
)
grid_clf_params

In [ ]:
grid_clf_search = GridSearchCV(
    grid_clf_pipeline,
    grid_clf_params,
    cv=cv,
    scoring="f1_macro",
    n_jobs=n_jobs,
    verbose=5,
)

In [ ]:
grid_clf_search.fit(X_svd_grid, y_train)

In [ ]:
grid_clf_search.best_params_

In [ ]:
wandb_log_search_results(grid_clf_search, "hyperparams_grid_search_clf", "hyperparams_grid_search")

In [ ]:
wandb_save_model(
    grid_clf_search.best_estimator_,
    os.path.join(MODEL_DIR, "best_model_after_grid_search"),
)

# Result

In [ ]:
best_params_res = {
    **{f"features__{k}": v for k, v in best_tfidf_word.items()},
    **{f"features__{k}": v for k, v in best_tfidf_char.items()},
    **best_svd_params,
    **grid_clf_search.best_params_,
}

Copy this params and paste them into training notebook to train the final model with the best params. 

In [ ]:
best_params_res